# 03 — Homology

In the previous notebook we studied simplicial complexes. We now add algebraic structure to them.

The central idea is:

> **Homology detects holes.**

We work first over the field

$$
\mathbb F_2=\{0,1\},
$$

where

$$
1+1=0.
$$

This removes orientation signs and lets us focus on the main ideas.

### Learning goals

We will define chains, boundaries, cycles, homology groups, and Betti numbers; compute them for simple complexes; and see concretely why a loop becomes trivial when it is filled by a triangle.

## 1. Chains

Let $K$ be a simplicial complex. The **$k$-chain group** over $\mathbb F_2$ is the vector space generated by the $k$-simplices:

$$
C_k(K;\mathbb F_2).
$$

If the edges are $e_1,e_2,e_3$, then a $1$-chain has the form

$$
c=a_1e_1+a_2e_2+a_3e_3,
\qquad a_i\in\mathbb F_2.
$$

Because

$$
1+1=0,
$$

a simplex appearing twice cancels.

Thus a chain can be represented computationally by the set of simplices whose coefficient is $1$.

### Pseudocode

```text
Represent a chain by its simplices with coefficient 1.

To add two chains:
    take the symmetric difference.
```

In [1]:
def add_chains(chain_a, chain_b):
    return set(chain_a) ^ set(chain_b)

e1 = frozenset({"a", "b"})
e2 = frozenset({"b", "c"})
e3 = frozenset({"c", "d"})

add_chains({e1, e2}, {e2, e3})


{frozenset({'a', 'b'}), frozenset({'c', 'd'})}

## 2. The boundary operator

The **boundary map** is

$$
\partial_k:C_k(K)\longrightarrow C_{k-1}(K).
$$

Over $\mathbb F_2$, the boundary of a simplex is the sum of its codimension-one faces.

For an edge,

$$
\partial[a,b]=[a]+[b].
$$

For a triangle,

$$
\partial[a,b,c]
=
[a,b]+[a,c]+[b,c].
$$

For a tetrahedron,

$$
\partial[a,b,c,d]
=
[a,b,c]+[a,b,d]+[a,c,d]+[b,c,d].
$$

The crucial identity is

$$
\boxed{\partial^2=0.}
$$

### Pseudocode

```text
INPUT: a k-simplex sigma

GENERATE: all codimension-one faces of sigma

RETURN: their sum over F_2
```

In [2]:
from itertools import combinations
import numpy as np

def codimension_one_faces(simplex):
    simplex = frozenset(simplex)
    if len(simplex) <= 1:
        return set()
    return {frozenset(f) for f in combinations(simplex, len(simplex)-1)}

def boundary_of_simplex(simplex):
    return codimension_one_faces(simplex)

boundary_of_simplex({"a", "b", "c"})


{frozenset({'a', 'c'}), frozenset({'a', 'b'}), frozenset({'b', 'c'})}

## 3. Boundaries of chains and the identity $\partial^2=0$

The boundary of a chain is obtained by adding the boundaries of its simplices. Repeated faces cancel over $\mathbb F_2$.

### Pseudocode

```text
boundary_of_chain(c):
    boundary = empty chain
    for simplex in c:
        boundary = boundary + boundary(simplex)
    return boundary
```

In [3]:
def boundary_of_chain(chain):
    boundary = set()
    for simplex in chain:
        boundary = add_chains(boundary, boundary_of_simplex(simplex))
    return boundary

triangle = frozenset({"a", "b", "c"})
edge_cycle = boundary_of_chain({triangle})

print("Boundary:", edge_cycle)
print("Boundary of boundary:", boundary_of_chain(edge_cycle))


Boundary: {frozenset({'c', 'a'}), frozenset({'b', 'a'}), frozenset({'c', 'b'})}
Boundary of boundary: set()


The second output is the zero chain. Hence

$$
\partial_1\partial_2=0.
$$

This is the algebraic reason that every boundary is automatically a cycle.

## 4. Cycles and boundaries

A $k$-chain $c$ is a **cycle** if

$$
\partial_k c=0.
$$

The cycle group is

$$
Z_k(K)=\ker\partial_k.
$$

A $k$-chain is a **boundary** if it is the boundary of a $(k+1)$-chain:

$$
B_k(K)=\operatorname{im}\partial_{k+1}.
$$

Because

$$
\partial_k\partial_{k+1}=0,
$$

we have

$$
\boxed{B_k(K)\subseteq Z_k(K).}
$$

### Example

The three edges of a triangle form a $1$-cycle.

### Non-example

A single edge is generally not a cycle because

$$
\partial[a,b]=[a]+[b]\neq0.
$$

## 5. Homology

The $k$-th homology group is

$$
H_k(K;\mathbb F_2)
=
Z_k(K)/B_k(K).
$$

Equivalently,

$$
H_k(K;\mathbb F_2)
=
\frac{\ker\partial_k}{\operatorname{im}\partial_{k+1}}.
$$

So homology identifies cycles that differ by a boundary.

This gives the fundamental interpretation:

$$
\boxed{
\text{homology}=
\frac{\text{cycles}}{\text{boundaries}}
}
$$

A cycle is a candidate hole; if it is the boundary of a higher-dimensional simplex or chain, it does not represent a homology class.

The chain-complex picture is

$$
C_{k+1}
\overset{\partial_{k+1}}{\longrightarrow}
C_k
\overset{\partial_k}{\longrightarrow}
C_{k-1},
$$

with

$$
\operatorname{im}\partial_{k+1}\subseteq\ker\partial_k.
$$

## 6. Boundary matrices

Choose an ordering of the simplices. The boundary map becomes a matrix.

For the triangle with vertices $a,b,c$ and edges

$$
e_0=[a,b],\quad e_1=[a,c],\quad e_2=[b,c],
$$

the matrix for $\partial_1$ is

$$
D_1=
\begin{pmatrix}
1&1&0\\\\
1&0&1\\\\
0&1&1
\end{pmatrix}.
$$

### Pseudocode

```text
INPUT:
    ordered k-simplices
    ordered (k-1)-simplices

CREATE:
    zero matrix D

FOR each k-simplex:
    FOR each codimension-one face:
        put 1 in the corresponding matrix entry

RETURN D
```

In [4]:
def boundary_matrix(higher_simplices, lower_simplices):
    lower_index = {s: i for i, s in enumerate(lower_simplices)}
    D = np.zeros((len(lower_simplices), len(higher_simplices)), dtype=np.uint8)
    for j, simplex in enumerate(higher_simplices):
        for face in codimension_one_faces(simplex):
            D[lower_index[face], j] = 1
    return D

vertices = [frozenset({"a"}), frozenset({"b"}), frozenset({"c"})]
edges = [
    frozenset({"a", "b"}),
    frozenset({"a", "c"}),
    frozenset({"b", "c"}),
]

D1 = boundary_matrix(edges, vertices)
D1


array([[1, 1, 0],
       [1, 0, 1],
       [0, 1, 1]], dtype=uint8)

In [5]:
def matmul_mod2(A, x):
    return (A @ x) % 2

cycle_vector = np.array([1, 1, 1], dtype=np.uint8)
print(matmul_mod2(D1, cycle_vector))


[0 0 0]


The zero vector shows that the sum of the three edges is a cycle:

$$
[a,b]+[a,c]+[b,c]\in Z_1.
$$

Now add the $2$-simplex $[a,b,c]$. Its boundary matrix is

$$
D_2=
\begin{pmatrix}
1\\\\1\\\\1
\end{pmatrix}.
$$

In [6]:
triangles = [frozenset({"a", "b", "c"})]
D2 = boundary_matrix(triangles, edges)

print("D2 =")
print(D2)
print("\nD1 D2 mod 2 =")
print((D1 @ D2) % 2)


D2 =
[[1]
 [1]
 [1]]

D1 D2 mod 2 =
[[0]
 [0]
 [0]]


Since

$$
D_1D_2=0,
$$

we see $\partial^2=0$ at the matrix level.

More importantly, the edge cycle is in the image of $D_2$. Therefore it is a boundary, and

$$
H_1(\text{filled triangle};\mathbb F_2)=0.
$$

If we remove the $2$-simplex, there are no $2$-boundaries, so the same cycle becomes a nontrivial class:

$$
H_1(\partial\Delta^2;\mathbb F_2)\cong\mathbb F_2.
$$

## 7. Betti numbers

The **Betti number** is

$$
\beta_k=\dim H_k.
$$

Using rank-nullity,

$$
\dim\ker\partial_k
=
\dim C_k-\operatorname{rank}D_k.
$$

Therefore,

$$
\boxed{
\beta_k
=
\dim C_k
-\operatorname{rank}D_k
-\operatorname{rank}D_{k+1}.
}
$$

Geometrically:

$$
\beta_0=\text{number of connected components},
$$

while $\beta_1,\beta_2,\ldots$ count independent holes in successive dimensions.

### Pseudocode

```text
beta_k = dimension(C_k)
         - rank(D_k)
         - rank(D_{k+1})
```

In [7]:
def rank_mod2(A):
    A = np.array(A, dtype=np.uint8).copy()
    rows, cols = A.shape
    rank = 0

    for col in range(cols):
        pivot = next((r for r in range(rank, rows) if A[r, col]), None)
        if pivot is None:
            continue
        if pivot != rank:
            A[[rank, pivot]] = A[[pivot, rank]]
        for r in range(rows):
            if r != rank and A[r, col]:
                A[r] ^= A[rank]
        rank += 1
        if rank == rows:
            break
    return rank

def betti_number(dim_chain, Dk, Dkp1=None):
    r_next = 0 if Dkp1 is None else rank_mod2(Dkp1)
    return dim_chain - rank_mod2(Dk) - r_next

print("rank(D1) =", rank_mod2(D1))
print("rank(D2) =", rank_mod2(D2))
print("filled triangle beta_1 =", betti_number(3, D1, D2))
print("boundary triangle beta_1 =",
      betti_number(3, D1, np.zeros((3, 0), dtype=np.uint8)))


rank(D1) = 2
rank(D2) = 1
filled triangle beta_1 = 0
boundary triangle beta_1 = 1


## 8. Zeroth homology

$H_0$ detects connected components.

Since

$$
\partial_0=0,
$$

every vertex is a $0$-cycle. The boundaries of edges identify vertices that lie in the same connected component.

Thus

$$
\boxed{\beta_0=\text{number of connected components}.}
$$

For two isolated points,

$$
\beta_0=2.
$$

For a connected interval,

$$
\beta_0=1.
$$

In [8]:
isolated_vertices = [frozenset({"a"}), frozenset({"b"})]
D1_isolated = np.zeros((2, 0), dtype=np.uint8)

path_vertices = [
    frozenset({"a"}), frozenset({"b"}), frozenset({"c"})
]
path_edges = [
    frozenset({"a", "b"}), frozenset({"b", "c"})
]
D1_path = boundary_matrix(path_edges, path_vertices)

print("Two isolated points: beta_0 =",
      betti_number(2, D1_isolated))
print("Connected path: beta_0 =",
      betti_number(3, D1_path))


Two isolated points: beta_0 = 2
Connected path: beta_0 = 1


## 9. The circle

Consider the simplicial complex

$$
a-b-c-a.
$$

It has one connected component and one independent loop, so we expect

$$
\beta_0=1,\qquad\beta_1=1.
$$

There are no $2$-simplices, hence $D_2$ is the empty matrix with three rows.

In [9]:
circle_vertices = [
    frozenset({"a"}), frozenset({"b"}), frozenset({"c"})
]
circle_edges = [
    frozenset({"a", "b"}),
    frozenset({"b", "c"}),
    frozenset({"a", "c"})
]

D1_circle = boundary_matrix(circle_edges, circle_vertices)
D2_circle = np.zeros((3, 0), dtype=np.uint8)

print("beta_0 =", betti_number(3, D1_circle))
print("beta_1 =", betti_number(3, D1_circle, D2_circle))


beta_0 = 1
beta_1 = 1


Thus

$$
\beta_0=1,\qquad\beta_1=1.
$$

This is the simplest computational example of a nontrivial $H_1$ class.

The same three edges become homologically trivial when the triangle is filled in.

## 10. The boundary of a tetrahedron

The boundary of a tetrahedron is a triangulation of the $2$-sphere.

It has

$$
f_0=4,\qquad f_1=6,\qquad f_2=4.
$$

We expect

$$
\beta_0=1,\qquad
\beta_1=0,\qquad
\beta_2=1.
$$

The nontrivial $H_2$ class is represented by the sum of the four triangular faces.

In [10]:
tetra_vertices = [frozenset({v}) for v in "abcd"]
tetra_edges = [frozenset(e) for e in combinations("abcd", 2)]
tetra_faces = [frozenset(f) for f in combinations("abcd", 3)]

D1_tetra = boundary_matrix(tetra_edges, tetra_vertices)
D2_tetra = boundary_matrix(tetra_faces, tetra_edges)
D3_tetra = np.zeros((len(tetra_faces), 0), dtype=np.uint8)

beta0 = betti_number(len(tetra_vertices), D1_tetra)
beta1 = betti_number(len(tetra_edges), D1_tetra, D2_tetra)
beta2 = betti_number(len(tetra_faces), D2_tetra, D3_tetra)

print("beta_0 =", beta0)
print("beta_1 =", beta1)
print("beta_2 =", beta2)


beta_0 = 1
beta_1 = 0
beta_2 = 1


## 11. Euler characteristic

The Euler characteristic is

$$
\chi(K)=f_0-f_1+f_2-f_3+\cdots.
$$

It also satisfies

$$
\boxed{
\chi(K)=\beta_0-\beta_1+\beta_2-\beta_3+\cdots.
}
$$

For the circle,

$$
\chi=3-3=0
$$

and

$$
\chi=1-1=0.
$$

For the filled triangle,

$$
\chi=3-3+1=1.
$$

For the tetrahedral sphere,

$$
\chi=4-6+4=2.
$$

In [11]:
def simplex_counts(complex_):
    from collections import Counter
    return dict(sorted(Counter(len(s)-1 for s in complex_).items()))

def euler_characteristic(complex_):
    return sum((-1)**k * n for k, n in simplex_counts(complex_).items())

circle_complex = set(circle_vertices + circle_edges)
filled_triangle = set(vertices + edges + triangles)
tetra_boundary = set(tetra_vertices + tetra_edges + tetra_faces)

print("Circle:", euler_characteristic(circle_complex))
print("Filled triangle:", euler_characteristic(filled_triangle))
print("Tetrahedron boundary:", euler_characteristic(tetra_boundary))


Circle: 0
Filled triangle: 1
Tetrahedron boundary: 2


## 12. Why $\mathbb F_2$?

Homology can also be defined over $\mathbb Z$, $\mathbb Q$, $\mathbb R$, and other coefficient fields.

Over the integers, orientations matter. For example,

$$
\partial[a,b]=[b]-[a]
$$

and

$$
\partial[a,b,c]
=
[b,c]-[a,c]+[a,b].
$$

Over $\mathbb F_2$,

$$
-1=1,
$$

so all signs disappear.

Thus $\mathbb F_2$ is a convenient first coefficient system for computational experiments. Later we can return to orientations and integer coefficients.

## 13. The central picture

A simplicial complex produces a chain complex

$$
\cdots
\longrightarrow
C_2
\overset{\partial_2}{\longrightarrow}
C_1
\overset{\partial_1}{\longrightarrow}
C_0
\longrightarrow0.
$$

The identity

$$
\partial_k\partial_{k+1}=0
$$

implies

$$
\operatorname{im}\partial_{k+1}
\subseteq
\ker\partial_k.
$$

Therefore we can form

$$
H_k=
\frac{\ker\partial_k}
{\operatorname{im}\partial_{k+1}}.
$$

The first three notebooks now form the conceptual chain

$$
\boxed{
\text{metric space}
\longrightarrow
\text{simplicial complex}
\longrightarrow
\text{homology}.
}
$$

Next we will add **scale** through filtrations, leading eventually to persistent homology.

## 14. Summary

A **$k$-chain** is a formal sum of $k$-simplices.

The boundary map is

$$
\partial_k:C_k\to C_{k-1}.
$$

It satisfies

$$
\partial^2=0.
$$

The cycle group is

$$
Z_k=\ker\partial_k,
$$

and the boundary group is

$$
B_k=\operatorname{im}\partial_{k+1}.
$$

Homology is

$$
H_k=Z_k/B_k.
$$

The Betti number is

$$
\beta_k=\dim H_k.
$$

In low dimensions:

$$
\beta_0=\text{connected components},
$$

$$
\beta_1=\text{independent one-dimensional holes},
$$

$$
\beta_2=\text{independent two-dimensional holes}.
$$

Finally,

$$
\chi
=
\sum_k(-1)^kf_k
=
\sum_k(-1)^k\beta_k.
$$

## Exercises

### Exercise 1 — Triangle

Construct the boundary of

$$
\{a,b,c\}
$$

and verify

$$
\partial^2=0.
$$

### Exercise 2 — Filled versus unfilled triangle

Compute the Betti numbers of the triangle boundary and the filled triangle. Explain why adding the $2$-simplex changes $\beta_1$.

### Exercise 3 — Two circles

Construct two disconnected triangular loops. Predict

$$
\beta_0
\quad\text{and}\quad
\beta_1
$$

before computing them.

### Exercise 4 — Tetrahedral sphere

Verify

$$
\beta_0=1,\qquad
\beta_1=0,\qquad
\beta_2=1.
$$

Check the Euler characteristic.

### Exercise 5 — Two independent loops

Construct a graph containing two independent loops. Find two independent $1$-cycles and predict $\beta_1$.

### Exercise 6 — Filling a loop

Start with a loop and add a $2$-simplex whose boundary is that loop. Observe how $H_1$ changes.

### Exercise 7 — Challenge

Implement a function that returns all Betti numbers of a finite simplicial complex over $\mathbb F_2$.

Test it on:

- a point;
- an interval;
- a circle;
- a filled triangle;
- the boundary of a tetrahedron.

Predict the answer before running the code.

## References

1. H. Edelsbrunner and J. Harer, *Computational Topology: An Introduction*, American Mathematical Society, 2010.

2. A. Hatcher, *Algebraic Topology*, Cambridge University Press, 2002.

3. F. Chazal and B. Michel, *An Introduction to Topological Data Analysis: Fundamental and Practical Aspects for Data Scientists*, *Frontiers in Artificial Intelligence* 4 (2021), 667963.  
   https://doi.org/10.3389/frai.2021.667963
4. J. B. Fraleigh, *A First Course in Abstract Algebra (Chapter VIII: Groups in Topology)*, Pearson.